## **Практическое заятие 4**
автоматический разведочный анализ 

рассматриваем три новые библиотеки: `Dtale`, `ydata_profiling`, `sweetviz`

In [2]:
import dtale as Dtale
from ydata_profiling import ProfileReport
import sweetviz as sv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

/mnt/z/My_Coding/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_2346/2210242012.py:2: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


## Чтение датасета и первичный осмотр

In [4]:
data = pd.read_csv('Students.csv')

data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   gender                       1000 non-null   object
 1   race_ethnicity               1000 non-null   object
 2   parental_level_of_education  1000 non-null   object
 3   lunch                        1000 non-null   object
 4   test_preparation_course      1000 non-null   object
 5   math_score                   1000 non-null   int64 
 6   reading_score                1000 non-null   int64 
 7   writing_score                1000 non-null   int64 
dtypes: int64(3), object(5)
memory usage: 62.6+ KB


## 3. Загрузка и первичный осмотр

`Score` — сумма трёх экзаменов, как в работе 3. Её кладём в таблицу до отчётов, чтобы библиотеки посчитали связь общего балла с остальными полями.

In [5]:
data['Score'] = data[['math_score', 'reading_score', 'writing_score']].sum(axis=1)

print('размер датасета: ', data.shape)

размер датасета:  (1000, 9)


In [6]:
print(data.head(10))
print()
data.info()
print("\nпропуски:", int(data.isnull().sum().sum()))
print("дубликатов строк:", int(data.duplicated().sum()))

   gender race_ethnicity parental_level_of_education         lunch  \
0  female        group B           bachelor's degree      standard   
1  female        group C                some college      standard   
2  female        group B             master's degree      standard   
3    male        group A          associate's degree  free/reduced   
4    male        group C                some college      standard   
5  female        group B          associate's degree      standard   
6  female        group B                some college      standard   
7    male        group B                some college  free/reduced   
8    male        group D                 high school  free/reduced   
9  female        group B                 high school  free/reduced   

  test_preparation_course  math_score  reading_score  writing_score  Score  
0                    none          72             72             74    218  
1               completed          69             90             88    247 

## dtale describe | correlations 

`dtale.show` поднимает локальную страницу. В ноутбуке браузер не открываем: ниже те же числа, которые лежат на вкладках Describe и Correlations. После печати сервер гасится, чтобы не занимать порт.

In [8]:
view = Dtale.show(
    data,
    open_browser=False,
    notebook=False,
    subprocess=True,
    host='127.0.0.1'
)

print('dtale: ', view._main_url)

dtale:  http://127.0.0.1:40000/dtale/main/1


In [9]:
print('Describe, числовые поля: \n')
print(data.describe().T.round(2))

Describe, числовые поля: 

                count    mean    std   min     25%    50%    75%    max
math_score     1000.0   66.09  15.16   0.0   57.00   66.0   77.0  100.0
reading_score  1000.0   69.17  14.60  17.0   59.00   70.0   79.0  100.0
writing_score  1000.0   68.05  15.20  10.0   57.75   69.0   79.0  100.0
Score          1000.0  203.31  42.77  27.0  175.00  205.0  233.0  300.0


In [10]:
print("\nDescribe, категории:")
for col in ["gender", "race_ethnicity", "parental_level_of_education", "lunch", "test_preparation_course"]:
    print(f"\n{col}")
    print(data[col].value_counts())


Describe, категории:

gender
gender
female    518
male      482
Name: count, dtype: int64

race_ethnicity
race_ethnicity
group C    319
group D    262
group B    190
group E    140
group A     89
Name: count, dtype: int64

parental_level_of_education
parental_level_of_education
some college          226
associate's degree    222
high school           196
some high school      179
bachelor's degree     118
master's degree        59
Name: count, dtype: int64

lunch
lunch
standard        645
free/reduced    355
Name: count, dtype: int64

test_preparation_course
test_preparation_course
none         642
completed    358
Name: count, dtype: int64


In [11]:
print("\nCorrelations, Пирсон:")
print(data.corr(numeric_only=True).round(3))
view.kill()
print("\nсервер остановлен")

2026-10-24 11:44:41,311 - INFO     - Executing shutdown...
2026-10-24 11:44:41,312 - INFO     - Not running with the Werkzeug Server, exiting by searching gc for BaseWSGIServer



Correlations, Пирсон:
               math_score  reading_score  writing_score  Score
math_score          1.000          0.818          0.803  0.919
reading_score       0.818          1.000          0.955  0.970
writing_score       0.803          0.955          1.000  0.966
Score               0.919          0.970          0.966  1.000

сервер остановлен


In [12]:
data['Score'].mean()

203.312

Describe подтверждает работу 1: 1000 строк, пропусков нет, баллы лежат примерно от 0 до 100, среднее Score около 203.

Correlations повторяет работу 3. Чтение и письмо — 0,955. Математика с ними около 0,80–0,82. Score почти повторяет чтение и письмо (0,97), потому что он из них и состоит. Отрицательных связей между баллами нет.

## ydata-profiling

Отчёт пишется в два файла, как просит методичка: HTML для чтения и JSON для тех же чисел без вёрстки.

Разделы, которые смотрим:

- **Overview** — размер, пропуски, предупреждения;
- **Variables** — распределение каждого поля;
- **Correlations** — матрица, та же, что в D-Tale;
- **Interactions** — рассеяние пар числовых полей, те же облака, что строили вручную в работе 3.

In [15]:

import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)
REPORTS = Path("reports")
REPORTS.mkdir(exist_ok=True)


def association_with(report, target):
    series = pd.Series(report._associations[target]).drop(labels=[target])
    return series.reindex(series.abs().sort_values(ascending=False).index)


In [16]:
profile = ProfileReport(
    data,
    title="Students dataset",
    progress_bar=False,
    correlations={
        "auto": {"calculate": True},
        "pearson": {"calculate": True},
        "spearman": {"calculate": True},
    },
)
html_path = REPORTS / "students_profile.html"
json_path = REPORTS / "students_profile.json"
profile.to_file(html_path)
profile.to_file(json_path)
print("HTML:", html_path.resolve())
print("JSON:", json_path.resolve())

table = profile.description_set.table
print("\nOverview:")
print("строк", table["n"])
print("столбцов", table["n_var"])
print("пропущенных ячеек", table["n_cells_missing"])
print("дубликатов", table["n_duplicates"])

print("\nПредупреждения:")
for alert in profile.description_set.alerts:
    print("-", alert)

print("\nPearson:")
print(profile.description_set.correlations["pearson"].round(3))


100%|██████████| 9/9 [00:00<00:00, 89.99it/s]


HTML: reports/students_profile.html
JSON: reports/students_profile.json

Overview:
строк 1000
столбцов 9
пропущенных ячеек 0
дубликатов 0

Предупреждения:
- [Score] is highly overall correlated with [math_score] and 2 other fields
- [math_score] is highly overall correlated with [reading_score] and 2 other fields
- [reading_score] is highly overall correlated with [math_score] and 2 other fields
- [writing_score] is highly overall correlated with [math_score] and 2 other fields

Pearson:
               math_score  reading_score  writing_score  Score
math_score          1.000          0.818          0.803  0.919
reading_score       0.818          1.000          0.955  0.970
writing_score       0.803          0.955          1.000  0.966
Score               0.919          0.970          0.966  1.000



Пропусков и дубликатов нет, чистить таблицу перед выводами не нужно. Предупреждение о высокой корреляции — это пара чтение–письмо: библиотека видит то же узкое облако, что и ручной scatter.

По переменным баллы одномодальные, без второй вершины. Категории несбалансированы: master's degree всего 59 человек, group A — 89, standard lunch чаще, чем free/reduced. Поэтому крайние средние в работе 3 шумнее центральных.

Interactions для трёх экзаменов — восходящие полосы. Отдельной ветки «мальчики / девочки» на этих графиках нет: пол в этом отчёте категория, а не ось рассеяния. Разъезд средних по полу мы уже посчитали в работе 3, отчёт его не отменяет.

In [17]:
sweet = sv.analyze(data)
sweet_path = REPORTS / "students_sweetviz.html"
sweet.show_html(str(sweet_path), open_browser=False)
print("HTML:", sweet_path.resolve())

print("\nАссоциации со Score:")
print(association_with(sweet, "Score").round(3))


Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)


Report reports/students_sweetviz.html was generated.
HTML: reports/students_sweetviz.html

Ассоциации со Score:
reading_score                  0.970
writing_score                  0.966
math_score                     0.919
lunch                          0.290
test_preparation_course        0.257
parental_level_of_education    0.227
race_ethnicity                 0.188
gender                         0.131
dtype: float64


Sweetviz раскладывает Score по тем же связям. Сильнее всего его тянут сами экзамены, чтение и письмо впереди математики. Категории (обед, курс, образование, группа) дают заметно меньшие коэффициенты: они сдвигают среднее, но не заменяют связь между баллами.

## `Video_Games_Sales_as_at_22_Dec_2016.csv`

Одна библиотека, sweetviz.

`User_Score` в файле хранится текстом: у части игр стоит `tbd`, оценка ещё не выставлена. Такие ячейки переводятся в пропуск, иначе столбец не попадает в корреляции

In [18]:
games = pd.read_csv("Video_Games_Sales_as_at_22_Dec_2016.csv")
games["User_Score"] = pd.to_numeric(games["User_Score"], errors="coerce")
games = games.drop(columns=["Name", "Publisher", "Developer"])
print("размер:", games.shape)
print("пропуски:")
print(games.isna().sum())

game_report = sv.analyze(games)
game_path = REPORTS / "games_sweetviz.html"
game_report.show_html(str(game_path), open_browser=False)
print("HTML:", game_path.resolve())

print("\nСвязи с мировыми продажами:")
print(association_with(game_report, "Global_Sales").round(3))
print("\nПирсон:")
print(games.corr(numeric_only=True).round(3).to_string())


размер: (16719, 13)
пропуски:
Platform              0
Year_of_Release     269
Genre                 2
NA_Sales              0
EU_Sales              0
JP_Sales              0
Other_Sales           0
Global_Sales          0
Critic_Score       8582
Critic_Count       8582
User_Score         9129
User_Count         9129
Rating             6769
dtype: int64


Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)

Report reports/games_sweetviz.html was generated.
HTML: reports/games_sweetviz.html

Связи с мировыми продажами:
NA_Sales           0.941
EU_Sales           0.901
Other_Sales        0.749
JP_Sales           0.612
Critic_Count       0.304
User_Count         0.265
Critic_Score       0.245
Platform           0.190
Genre              0.111
Rating             0.098
User_Score         0.088
Year_of_Release   -0.076
dtype: float64

Пирсон:
                 Year_of_Release  NA_Sales  EU_Sales  JP_Sales  Other_Sales  Global_Sales  Critic_Score  Critic_Count  User_Score  User_Count
Year_of_Release            1.000    -0.093     0.004    -0.168        0.038        -0.076         0.011         0.223      -0.268       0.175
NA_Sales                  -0.093     1.000     0.765     0.450        0.639         0.941         0.241         0.295       0.086       0.246
EU_Sales                   0.004     0.765     1.000     0.435        0.723         0.901         0.221         0.278       0.055       0

In [19]:
games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16719 entries, 0 to 16718
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Platform         16719 non-null  object 
 1   Year_of_Release  16450 non-null  float64
 2   Genre            16717 non-null  object 
 3   NA_Sales         16719 non-null  float64
 4   EU_Sales         16719 non-null  float64
 5   JP_Sales         16719 non-null  float64
 6   Other_Sales      16719 non-null  float64
 7   Global_Sales     16719 non-null  float64
 8   Critic_Score     8137 non-null   float64
 9   Critic_Count     8137 non-null   float64
 10  User_Score       7590 non-null   float64
 11  User_Count       7590 non-null   float64
 12  Rating           9950 non-null   object 
dtypes: float64(10), object(3)
memory usage: 1.7+ MB


Продажи по регионам заполнены у всех 16719 строк. Оценок критиков нет у 8582 строк, это около половины. Пользовательская оценка после перевода `tbd` в пропуск отсутствует у 9129 строк, и ровно столько же нет числа отзывов. Возрастной рейтинг пуст у 6769 строк, год выпуска — у 269. По оценкам выводы можно делать только на заполненной части, а не на всей таблице.

Мировые продажи сильнее всего связаны с Северной Америкой (0,941) и Европой (0,901). `Global_Sales` собирается из региональных столбцов, поэтому такая связь ожидаема: Северная Америка даёт самую большую долю. Япония связана слабее (0,612).

Оценки критиков и игроков сходятся умеренно (0,581). Со сборами они связаны слабо: оценка критиков 0,245, оценка игроков 0,088. Высокий балл в этой таблице не означает большие продажи. Год выпуска со сборами почти не связан, коэффициент −0,076.

Платформа и жанр сдвигают продажи слабее региональных столбцов: ассоциации 0,190 и 0,111.